# SHAP Explainability — Street Segment Risk Classification

Explains **why** the logistic regression model assigns each street segment its risk class.

**Prerequisites:** Run `06_CityClassification.ipynb` first to produce a classified CSV for your target place.

**Model:** `LogisticRegression` (multi-class: `high`, `low`, `medium`)  
**Explainer:** `shap.LinearExplainer` — exact and fast for linear models  
**Output:** Dependence scatter · Waterfall · Summary · Force plots

## ⚙️ Configuration — Edit This Cell

In [ ]:
# ── EDIT HERE ────────────────────────────────────────────────────────────────
PLACE = "Raval, Barcelona, Spain"   # Must match a classified CSV already in csv/
# e.g. "Eixample, Barcelona, Spain"
# e.g. "De Pijp, Amsterdam, Netherlands"

# Index of the segment to spotlight in waterfall / force plots
SAMPLE_IDX = 0
# ─────────────────────────────────────────────────────────────────────────────

import os
BASE_DIR   = r'C:\Users\maria\Documents\GitHub\MaCAD26-G01-DataEncoding'
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
OUTPUT_DIR = os.path.join(BASE_DIR, 'csv')

place_slug = PLACE.replace(',', '').replace(' ', '_').lower()
CSV_PATH   = os.path.join(OUTPUT_DIR, f'{place_slug}_classified.csv')

print(f'Target place : {PLACE}')
print(f'CSV path     : {CSV_PATH}')
print(f'Model dir    : {MODEL_DIR}')

## Step 1: Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

shap.initjs()   # enables JS-based force plots in notebook
print('✓ Imports OK')

## Step 2: Load Model, Scaler & Classified Data

In [ ]:
# Load trained artefacts
scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
model  = joblib.load(os.path.join(MODEL_DIR, 'logistic_regression.pkl'))

print('✓ Scaler loaded')
print('✓ Model loaded :', type(model).__name__)
print('  Classes      :', list(model.classes_))

# Load the classified CSV produced by 06_CityClassification
if not Path(CSV_PATH).exists():
    raise FileNotFoundError(
        f'No classified CSV found for "{PLACE}".\n'
        f'Run 06_CityClassification.ipynb with PLACE = "{PLACE}" first.\n'
        f'Expected: {CSV_PATH}'
    )

df = pd.read_csv(CSV_PATH)
print(f'\n✓ Loaded {len(df):,} segments from {CSV_PATH}')
print(df.head(3))

## Step 3: Prepare Feature Matrix

In [ ]:
FEATURE_COLS = [
    'lighting_norm',
    'visibility_norm',
    'connectivity_norm',
    'enclosure_norm',
    'dominant_land_use_score_norm',
    'public_transport_proximity_m_norm',
]

# Human-readable labels for plots
FEATURE_LABELS = [
    'Lighting',
    'Visibility',
    'Connectivity',
    'Enclosure',
    'Land Use',
    'Transit Proximity',
]

X      = df[FEATURE_COLS].fillna(0).values
X_scaled = scaler.transform(X)

# DataFrame version for plot labels
X_df        = pd.DataFrame(X_scaled, columns=FEATURE_LABELS)
classes     = list(model.classes_)          # ['high', 'low', 'medium']

print(f'✓ Feature matrix: {X_df.shape}')
print(X_df.describe().round(3))

## Step 4: Build SHAP Explainer

`shap.LinearExplainer` is exact for logistic/linear models — no sampling needed, no speed trade-off.  
It returns one SHAP matrix per class (`shap_values` has shape `(n_samples, n_features, n_classes)`).

In [ ]:
explainer   = shap.LinearExplainer(model, X_scaled, feature_perturbation='correlation_dependent')
shap_values = explainer(X_scaled)   # Explanation object: .values shape = (n, features, classes)

print('✓ SHAP values computed')
print('  shap_values.values shape:', shap_values.values.shape)
print('  Classes order            :', classes)

## Step 5: Dependence Scatter Plots

**How to read:** X-axis = normalised feature value; Y-axis = SHAP contribution to that class.  
A downward slope means higher feature value → lower probability of that class, and vice versa.

In [ ]:
# Plot dependence scatter for each feature × each class
for class_idx, class_name in enumerate(classes):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(f'Dependence Scatter — Class: {class_name.upper()} risk  |  {PLACE}', fontsize=13)

    for feat_idx, (feat_col, feat_label) in enumerate(zip(FEATURE_COLS, FEATURE_LABELS)):
        ax = axes[feat_idx // 3][feat_idx % 3]
        ax.scatter(
            X_df[feat_label],
            shap_values.values[:, feat_idx, class_idx],
            alpha=0.5, s=15, c=shap_values.values[:, feat_idx, class_idx],
            cmap='coolwarm'
        )
        ax.axhline(0, color='grey', linewidth=0.7, linestyle='--')
        ax.set_xlabel(feat_label, fontsize=9)
        ax.set_ylabel('SHAP value', fontsize=9)
        ax.set_title(feat_label, fontsize=10)

    plt.tight_layout()
    plt.show()

## Step 6: Summary Plot (Bar — Global Feature Importance)

Mean absolute SHAP value per feature across all segments and classes.  
Shows which features drive the model most overall.

In [ ]:
# shap.summary_plot expects (n_samples, n_features) — average across classes
shap_mean = shap_values.values.mean(axis=2)  # (n_samples, n_features)

shap.summary_plot(
    shap_mean,
    X_df,
    plot_type='bar',
    show=True
)
plt.title(f'Global Feature Importance — {PLACE}', pad=12)

## Step 7: Summary Plot (Dot — Per-Class Feature Effects)

One dot per segment. Colour = feature value (blue=low, red=high).  
X-axis spread shows direction and magnitude of effect on each class.

In [ ]:
for class_idx, class_name in enumerate(classes):
    print(f'\n--- Class: {class_name.upper()} risk ---')
    shap.summary_plot(
        shap_values.values[:, :, class_idx],
        X_df,
        plot_type='dot',
        show=True
    )

## Step 8: Waterfall Plot — Single Segment

**How to read:** Starting from the base value (average model output), each bar shows how much one feature *pushes* the prediction up (red) or down (blue) for the predicted class of this segment.  

Change `SAMPLE_IDX` at the top to inspect any segment.

In [ ]:
predicted_class     = df['risk_class'].iloc[SAMPLE_IDX]
predicted_class_idx = classes.index(predicted_class)

print(f'Segment {SAMPLE_IDX} — predicted class: {predicted_class.upper()} risk')
print(f'(showing SHAP values for class index {predicted_class_idx}: "{predicted_class}")')

exp_single = shap.Explanation(
    values       = shap_values.values[SAMPLE_IDX, :, predicted_class_idx],
    base_values  = explainer.expected_value[predicted_class_idx],
    data         = X_df.iloc[SAMPLE_IDX].values,
    feature_names= FEATURE_LABELS,
)

shap.plots.waterfall(exp_single)

## Step 9: Force Plot — Single Segment

Same information as the waterfall, but in a compact horizontal layout.  
Red features push the prediction higher; blue push it lower.

In [ ]:
shap.force_plot(
    base_value    = explainer.expected_value[predicted_class_idx],
    shap_values   = shap_values.values[SAMPLE_IDX, :, predicted_class_idx],
    features      = X_df.iloc[SAMPLE_IDX],
    feature_names = FEATURE_LABELS,
    matplotlib    = True
)

## Step 10: Compare SHAP Profiles Across Risk Classes

Mean SHAP value per feature for each risk class.  
Reveals which features most distinguish, say, *high* from *low* risk segments.

In [ ]:
# Build a DataFrame: mean SHAP per class per feature
rows = []
for class_idx, class_name in enumerate(classes):
    mean_shap = shap_values.values[:, :, class_idx].mean(axis=0)
    rows.append(dict(zip(FEATURE_LABELS, mean_shap)) | {'class': class_name})

shap_profile = pd.DataFrame(rows).set_index('class')
print('Mean SHAP values per class:')
print(shap_profile.round(4))

# Grouped bar chart
ax = shap_profile.T.plot(
    kind='bar',
    figsize=(11, 5),
    color=['#e74c3c', '#2ecc71', '#f39c12'],
    edgecolor='white',
    width=0.7
)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_title(f'Mean SHAP per Feature × Class — {PLACE}', fontsize=12)
ax.set_ylabel('Mean SHAP value')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Risk class')
plt.tight_layout()
plt.show()

## Step 11: Export SHAP Values to CSV

One row per segment; one column per (feature × class) combination.  
Useful for mapping SHAP values spatially in QGIS or further analysis.

In [ ]:
shap_rows = []
for class_idx, class_name in enumerate(classes):
    tmp = pd.DataFrame(
        shap_values.values[:, :, class_idx],
        columns=[f'shap_{feat}_{class_name}' for feat in FEATURE_LABELS]
    )
    shap_rows.append(tmp)

shap_df = pd.concat(shap_rows, axis=1)

# Attach osmid and predicted class if present
for col in ['osmid', 'risk_class', 'risk_probability']:
    if col in df.columns:
        shap_df.insert(0, col, df[col].values)

out_path = os.path.join(OUTPUT_DIR, f'{place_slug}_shap_values.csv')
shap_df.to_csv(out_path, index=False)
print(f'✓ SHAP values exported to: {out_path}')
print(f'  Columns: {list(shap_df.columns[:8])} ...')
shap_df.head(3)